In [0]:
%run "../00-common/01.environment-config"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
fact_path = f"{s3_root_path}/Gold/Aaisha/fact_session_results"

drivers_path = f"{s3_root_path}/Gold/Aaisha/dim_drivers"

target_path = f"{s3_root_path}/Gold/Aaisha/driver_standings"

print("Fact Path    :", fact_path)
print("Drivers Path :", drivers_path)
print("Target Path  :", target_path)

In [0]:
fact_df = (
    spark.read
        .format("delta")
        .load(fact_path)
)

drivers_df = (
    spark.read
        .format("delta")
        .load(drivers_path)
)

In [0]:
display(fact_df)
display(drivers_df)

In [0]:
driver_points_df = (
    fact_df
        .groupBy(
            "season",
            "driverId"
        )
        .agg(
            F.sum("points").alias("total_points")
        )
)

display(driver_points_df)

In [0]:
window_spec = (
    Window
        .partitionBy("season")
        .orderBy(F.desc("total_points"))
)

In [0]:
driver_standings_df = (
    driver_points_df
        .withColumn(
            "driver_rank",
            F.dense_rank().over(window_spec)
        )
)

display(driver_standings_df)

In [0]:
driver_standings_df = (
    driver_standings_df
        .withColumnRenamed("driverId", "driver_id")
        .join(
            drivers_df.select(
                "driver_id",
                "driver_name"
            ),
            "driver_id",
            "left"
        )
)

display(driver_standings_df)

In [0]:
driver_standings_final_df = (
    driver_standings_df
        .select(
            "season",
            "driver_rank",
            "driver_id",
            "driver_name",
            "total_points"
        )
        .orderBy(
            "season",
            "driver_rank"
        )
)

display(driver_standings_final_df)

In [0]:
target_path = f"{s3_root_path}/Gold/Aaisha/driver_standings"

(
    driver_standings_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

print("Driver standings successfully created")

In [0]:
driver_standings_check = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(driver_standings_check)